# Results paragraph: Longitudinal harmonization and batch effects

Uses assembled **batch-effect** outputs (ΔR²adj = full model minus reduced model without **batch device/software**) to fill the longitudinal harmonization results paragraph—aligned with Figure 3 when using pooled batch effects. Data: `data/batch_effects/batch_effects_all_outputs.rds` (rows with `output_type == "batch_effects"`). Filter: `qc_covariate == "no_quality"`, five metrics (MKT, ICVF, RTOP, FA, MD).

**Figure 3:** Panel A = raw batch-effect heatmap; Panels B/C = mean batch effects before vs after harmonization when the batch_effects pipeline is used.

## Setup and load batch effects

In [7]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

batch_rds <- fs::path(project_root, "data", "batch_effects", "batch_effects_all_outputs.rds")
if (!file.exists(batch_rds)) {
  stop("Batch effects file not found: ", batch_rds, "\nRun assemble_batch_effects.R after calculate_batch_effects.R")
}

df_batch <- readRDS(batch_rds)
if (!is.data.frame(df_batch)) stop("Batch effects file is not a data.frame.")
if ("output_type" %in% names(df_batch)) {
  df_batch <- df_batch %>% filter(output_type == "batch_effects")
}
if (nrow(df_batch) == 0) stop("No pooled batch_effects rows in ", batch_rds)

qc_target <- "no_quality"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT",
  NODDI_icvf = "ICVF",
  MAPMRI_rtop = "RTOP",
  GQI_fa = "FA",
  GQI_md = "MD"
)

df <- df_batch %>%
  filter(metric %in% metrics_keep, !is.na(effect_size))
if ("qc_covariate" %in% names(df) && qc_target %in% df$qc_covariate) {
  df <- df %>% filter(qc_covariate == qc_target)
}
cat("[batch_effects] N rows (no_quality, five metrics):", nrow(df), "\n")

[batch_effects] N rows (no_quality, five metrics): 620 


## Extract numbers for paragraph

In [8]:
# Raw data only (univariate level, Fig 3a)
raw <- df %>% filter(source == "raw")
if (nrow(raw) == 0) stop("No raw batch effects found.")

# Range of batch effects (raw): "These effects ranged between X-X" (as %)
effect_pct_min <- min(raw$effect_size, na.rm = TRUE) * 100
effect_pct_max <- max(raw$effect_size, na.rm = TRUE) * 100
range_lo <- round(effect_pct_min, 1)
range_hi <- round(effect_pct_max, 1)

# Average batch effect (%) by metric (raw), for all five metrics
raw_by_metric <- raw %>%
  group_by(metric) %>%
  summarise(mean_effect_pct = mean(effect_size, na.rm = TRUE) * 100, .groups = "drop")

mkt_avg_pct <- raw_by_metric %>% filter(metric == "DKI_mkt") %>% pull(mean_effect_pct) %>% `[`(1)
rtop_avg_pct <- raw_by_metric %>% filter(metric == "MAPMRI_rtop") %>% pull(mean_effect_pct) %>% `[`(1)
mkt_avg_pct <- round(replace(mkt_avg_pct, is.na(mkt_avg_pct), 0), 1)
rtop_avg_pct <- round(replace(rtop_avg_pct, is.na(rtop_avg_pct), 0), 1)
metric_avg_order <- c("DKI_mkt", "GQI_fa", "NODDI_icvf", "GQI_md", "MAPMRI_rtop")
metric_avg_text <- raw_by_metric %>%
  mutate(metric_label = unname(metric_labels[metric]), mean_effect_pct = round(mean_effect_pct, 1)) %>%
  mutate(metric = factor(metric, levels = metric_avg_order)) %>%
  arrange(metric) %>%
  transmute(txt = if_else(
    metric_label == "MKT",
    sprintf("%s%% of variance on average across bundles", sprintf("%.1f", mean_effect_pct)),
    sprintf("%s (%s%%)", metric_label, sprintf("%.1f", mean_effect_pct))
  )) %>%
  pull(txt)
metric_followed_text <- paste(metric_avg_text[-1], collapse = ", ")

# Harmonized: average effect (%) by metric; paragraph says "reduced to <X% for all"
harm <- df %>% filter(source == "harmonized")
harm_by_metric <- harm %>%
  group_by(metric) %>%
  summarise(mean_effect_pct = mean(effect_size, na.rm = TRUE) * 100, .groups = "drop")

max_harm_avg <- if (nrow(harm_by_metric) > 0) max(harm_by_metric$mean_effect_pct, na.rm = TRUE) else 0
cap_pct <- ceiling(max_harm_avg)
if (!is.finite(cap_pct) || cap_pct < 1) cap_pct <- 1

cat("Raw effect range (%):", range_lo, "-", range_hi, "\n")
cat("Raw MKT avg %:", mkt_avg_pct, "\n")
cat("Raw RTOP avg %:", rtop_avg_pct, "\n")
cat("Raw metric averages %:", paste(c(metric_avg_text[1], metric_followed_text), collapse = "; followed by "), "\n")
cat("Harmonized max mean effect (%):", round(max_harm_avg, 2), "=> paragraph <", cap_pct, "%\n")

# Bundle-category susceptibility in raw data (for text: cerebellar/subcortical most susceptible)
category_top2_text <- "Cerebellar and subcortical fibers"
if ("bundle_category" %in% names(raw)) {
  cat_map <- c(
    "ProjectionBasalGanglia" = "subcortical",
    "Cerebellum" = "cerebellar"
  )
  cat_rank <- raw %>%
    mutate(cat_pretty = dplyr::coalesce(unname(cat_map[as.character(bundle_category)]), as.character(bundle_category))) %>%
    group_by(cat_pretty) %>%
    summarise(mean_effect_pct = mean(effect_size, na.rm = TRUE) * 100, .groups = "drop") %>%
    arrange(desc(mean_effect_pct))
  top2 <- tolower(cat_rank$cat_pretty[seq_len(min(2, nrow(cat_rank)))])
  if (!all(c("cerebellar", "subcortical") %in% top2)) {
    category_top2_text <- paste0(tools::toTitleCase(top2[1]), if (length(top2) > 1) paste0(" and ", tools::toTitleCase(top2[2]), " fibers") else " fibers")
  }
}

# Min, max, and average batch effect (%%) across bundles within each metric (raw and harmonized)
effect_summary <- df %>%
  mutate(effect_pct = effect_size * 100) %>%
  group_by(source, metric) %>%
  summarise(
    min_pct = min(effect_pct, na.rm = TRUE),
    max_pct = max(effect_pct, na.rm = TRUE),
    mean_pct = mean(effect_pct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(metric_label = unname(metric_labels[metric])) %>%
  mutate(across(c(min_pct, max_pct, mean_pct), ~ round(., 2)))

cat("\nBatch effect (%% variance) across bundles by metric:\n")
print(effect_summary)

Raw effect range (%): 2.9 - 69.8 
Raw MKT avg %: 37.2 
Raw RTOP avg %: 8.2 
Raw metric averages %: 37.2% for MKT, 26.6% for FA, 20.1% for ICVF, 18.4% for MD, and 8.2% for RTOP 
Harmonized max mean effect (%): -0.15 => paragraph < 1 %

Batch effect (%% variance) across bundles by metric:


# A tibble: 10 × 6
   source     metric      min_pct max_pct mean_pct metric_label
   <chr>      <chr>         <dbl>   <dbl>    <dbl> <chr>       
 1 harmonized DKI_mkt       -0.22   -0.12    -0.16 MKT         
 2 harmonized GQI_fa        -0.25   -0.2     -0.22 FA          
 3 harmonized GQI_md        -0.24   -0.17    -0.21 MD          
 4 harmonized MAPMRI_rtop   -0.25   -0.12    -0.17 RTOP        
 5 harmonized NODDI_icvf    -0.25   -0.09    -0.15 ICVF        
 6 raw        DKI_mkt       15.1    63.7     37.2  MKT         
 7 raw        GQI_fa         5.37   69.8     26.6  FA          
 8 raw        GQI_md         7.41   49.9     18.4  MD          
 9 raw        MAPMRI_rtop    2.87   26.0      8.21 RTOP        
10 raw        NODDI_icvf     3.54   61.7     20.1  ICVF        


## Filled paragraph

In [9]:
txt <- sprintf(
  paste(
    "In the unharmonized data, acquisition batch explained up to %s%% variance in the data (Figure 3a).",
    "MKT was the most susceptible to batch effects (Figure 3b,c), with batch explaining %s, followed by %s.",
    "%s tended to be most susceptible to batch effects.",
    "Following harmonization, batch effects were effectively eliminated for all measures (Figure 3b,c).",
    "Batch effects for all microstructural metrics are provided in Figure S10.",
    "These results demonstrate that longitudinal harmonization can be applied at scale and substantially attenuates scanner-related differences across acquisition environments.",
    sep = " "
  ),
  sprintf("%.1f", range_hi),
  metric_avg_text[1],
  metric_followed_text,
  category_top2_text
)
cat(txt, "\n")

In the unharmonized data, acquisition batch explained up to 69.8% variance in the data (Figure 3a). MKT was the most susceptible to batch effects (Figure 3b,c), with batch explaining 37.2% of variance on average across bundles, followed by FA (26.6%), ICVF (20.1%), MD (18.4%), and RTOP (8.2%). Cerebellar and subcortical fibers tended to be most susceptible to batch effects. Following harmonization, batch effects were effectively eliminated for all measures (Figure 3b,c). Batch effects for all microstructural metrics are provided in Figure S10. These results demonstrate that longitudinal harmonization can be applied at scale and substantially attenuates scanner-related differences across acquisition environments.
